In [ ]:
import json
import re
from fractions import Fraction
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# -----------------------------------------------------------------------------
# 1. Inputs, outputs and options
# -----------------------------------------------------------------------------
ROOT = Path("..")
MULTI_RUN = Path("../outputs/simulation")
GAIN_RUN = Path("../outputs/gain_curves")
OUTDIR = ROOT / "outputs" / "figures" / "fig3"
OUTDIR.mkdir(parents=True, exist_ok=True)

SPLIT_PREFIX = "main"
COOLING_MODE = "tmax_tmin"
DELTA_T_C = 1.0
BUDGET_SHARE = 0.20

HEAT_TYPES_A = ["any", "compound"]
HEAT_TYPE_BC = "any"
RULES_A = ["population", "density", "height", "random", "hwd_hist", "hwd_foresight"]
RULES_BC = ["oracle", "population", "density", "height", "random", "hwd_hist", "hwd_foresight"]
REFERENCE_RULES = {"hwd_foresight"}

OUT_STEM = OUTDIR / "fig3_policy"

# -----------------------------------------------------------------------------
# 2. Style
# -----------------------------------------------------------------------------
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams.update({
    "font.family": ["DejaVu Sans"],
    "font.size": 8,
    "axes.linewidth": 0.75,
    "xtick.major.width": 0.65,
    "ytick.major.width": 0.65,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.unicode_minus": False,
})
TEXT = "#202124"

RULE_LABELS = {
    "oracle": "Oracle",
    "population": "Population",
    "density": "Building density",
    "height": "Building height",
    "random": "Random",
    "hwd_hist": "Historical HWD",
    "hwd_foresight": "Concurrent HWD",
    "exposure_hist": "Historical exposure",
    "response_hist": "Historical cooling benefit",
}
RULE_COLORS = {
    "oracle": "#222222",
    "population": "#9C2E7F",
    "density": "#279A8D",
    "height": "#7A9A2E",
    "random": "#9A9A9A",
    "hwd_hist": "#2B7BB9",
    "hwd_foresight": "#E46F35",
    "exposure_hist": "#5B3F8C",
    "response_hist": "#7A6A2E",
}
HEAT_TYPE_LABELS = {"any": "Total exposure", "compound": "Compound heatwaves",
                    "day": "Daytime-only heatwaves", "night": "Nighttime-only heatwaves"}


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def resolve_run(path, marker):
    path = Path(path)
    require((path / marker).is_file(), f"{marker} not found in {path}")
    return path


def lighten(color, amount):
    rgb = np.array(mpl.colors.to_rgb(color))
    return tuple(rgb + (1.0 - rgb) * amount)


def style_axis(ax, grid_axis="y"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=TEXT, labelsize=7, length=3)
    if grid_axis:
        ax.grid(axis=grid_axis, color="#E8E8E8", linewidth=0.55, zorder=0)
    ax.set_axisbelow(True)


# -----------------------------------------------------------------------------
# 3. Load data
# -----------------------------------------------------------------------------
MULTI_RUN = resolve_run(MULTI_RUN, "multiyear_siting_efficiency.csv")
GAIN_RUN = resolve_run(GAIN_RUN, "gain_curves_exact_pooled.csv")
_gain_source = json.loads((GAIN_RUN / "summary.json").read_text(encoding="utf-8"))["multiyear_run"]
require(Path(_gain_source) == MULTI_RUN, f"The cooling-area curves were computed from a different simulation: {_gain_source}; re-run 06_simulation/02_gain_curves")


def load_rules():
    eff = pd.read_csv(MULTI_RUN / "multiyear_siting_efficiency.csv")
    splits = [s for s in eff["split"].unique() if s.startswith(SPLIT_PREFIX)]
    require(len(splits) == 1, f"Split name is not unique: {splits}")
    split = splits[0]
    years = [int(y) for y in re.findall(r"\d{4}", split)]
    use = eff[eff["split"].eq(split) & eff["heatwave_type"].isin(HEAT_TYPES_A)
              & eff["cooling_mode"].eq(COOLING_MODE) & np.isclose(eff["delta_t_C"], DELTA_T_C)
              & np.isclose(eff["budget_share"], BUDGET_SHARE) & eff["strategy"].isin(RULES_A)].copy()
    require(1 <= len(HEAT_TYPES_A) <= 2, "HEAT_TYPES_A must contain one or two heatwave types")
    require(set(HEAT_TYPES_A) <= set(use["heatwave_type"]), "HEAT_TYPES_A contains a heatwave type that is absent from the simulation")
    require(set(RULES_A) <= set(use["strategy"]), "RULES_A contains a rule that is absent from the simulation (building height needs the new multi-year simulation)")
    require(not use.duplicated(["city", "heatwave_type", "strategy"]).any(), "(a) Duplicate city-type-rule rows in the data")
    return use, split, years


def load_gain(split):
    def pick(frame):
        return frame[frame["split"].eq(split) & frame["heatwave_type"].eq(HEAT_TYPE_BC)
                     & frame["cooling_mode"].eq(COOLING_MODE) & frame["strategy"].isin(RULES_BC)]

    exact = pick(pd.read_csv(GAIN_RUN / "gain_curves_exact_pooled.csv"))
    curves = exact[np.isclose(exact["delta_t_C"], DELTA_T_C)].copy()
    pooled = pick(pd.read_csv(GAIN_RUN / "gain_curves_pooled.csv"))
    magnitude = pooled[np.isclose(pooled["budget_share"], BUDGET_SHARE)].copy()
    match = pick(pd.read_csv(GAIN_RUN / "cells_needed_to_match_reference.csv"))
    match = match[np.isclose(match["delta_t_C"], DELTA_T_C)].copy()
    require(set(RULES_BC) <= set(curves["strategy"]), "RULES_BC contains a rule that is absent from the cooling-area curves")
    require(len(magnitude) and magnitude["delta_t_C"].nunique() >= 2, "Budget-level or cooling-magnitude data are incomplete")
    require(np.isclose(float(Fraction(str(match["reference_budget"].iloc[0]))), BUDGET_SHARE),
            "The reference budget of the post-processing does not match BUDGET_SHARE")
    return curves, magnitude, match


# -----------------------------------------------------------------------------
# 4. Panels
# -----------------------------------------------------------------------------
def draw_rules_panel(ax, rules_data):
    rng = np.random.default_rng(20261114)
    y_positions = np.arange(len(RULES_A))[::-1]
    single = len(HEAT_TYPES_A) == 1
    offsets = [0.0] if single else [0.20, -0.20]
    markers, tints = ["o", "D"][:len(HEAT_TYPES_A)], [0.0, 0.42][:len(HEAT_TYPES_A)]
    rows = []
    for y0, rule in zip(y_positions, RULES_A):
        color, ref = RULE_COLORS[rule], rule in REFERENCE_RULES
        for heat, dy, marker, tint in zip(HEAT_TYPES_A, offsets, markers, tints):
            g = rules_data[rules_data["strategy"].eq(rule) & rules_data["heatwave_type"].eq(heat)]
            values = 100 * g.loc[g["oracle_avoided"] > 0, "relative_efficiency"].dropna().to_numpy(float)
            pooled = 100 * g["avoided_exposure"].sum() / g["oracle_avoided"].sum()
            q10, q25, q50, q75, q90 = np.quantile(values, [0.10, 0.25, 0.50, 0.75, 0.90])
            y, shade = y0 + dy, lighten(color, tint)
            ax.scatter(values, y + rng.normal(0, 0.075 if single else 0.035, len(values)),
                       s=9 if single else 6.5, color=color, alpha=0.30, linewidth=0, zorder=2)
            ax.hlines(y, q10, q90, color=shade, linewidth=0.85, alpha=0.95, zorder=3)
            ax.hlines(y, q25, q75, color=shade, linewidth=4.0 if single else 3.3, alpha=0.95, zorder=4)
            ax.scatter(q50, y, s=34 if marker == "o" else 25, marker=marker,
                       color="white" if ref else color, edgecolor=color if ref else "white",
                       linewidth=1.1 if ref else 0.7, zorder=6)
            rows.append(dict(heatwave_type=heat, rule=rule, n_cities=len(values), q10=q10, q25=q25, median=q50,
                             q75=q75, q90=q90, pooled_share=pooled))
    ax.axvline(100, color="#6F6F6F", linestyle=(0, (3, 3)), linewidth=0.8, zorder=1)
    ax.set_xlim(-5, 108)
    ax.set_ylim(-0.60, y_positions.max() + 0.60)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xlabel("Share of oracle benefit captured (%)", fontsize=7.5)
    ax.set_yticks(y_positions)
    ax.set_yticklabels([RULE_LABELS[r] for r in RULES_A], fontsize=7.0)
    ax.tick_params(axis="y", length=0)
    style_axis(ax, grid_axis="x")
    grey = "#5F5F5F"
    if single:
        handles = [Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=grey, markeredgecolor="white",
                          markersize=5.2, label="City median"),
                   Line2D([0], [0], color=grey, linewidth=4.0, solid_capstyle="butt", label="IQR"),
                   Line2D([0], [0], color=grey, linewidth=0.85, label="10th\u201390th percentile")]
    else:
        handles = [Line2D([0], [0], color=lighten(grey, tint), linewidth=3.3, solid_capstyle="butt", marker=marker,
                          markerfacecolor=grey, markeredgecolor="white", markeredgewidth=0.7,
                          markersize=5.4 if marker == "o" else 4.6, label=HEAT_TYPE_LABELS[heat])
                   for heat, marker, tint in zip(HEAT_TYPES_A, markers, tints)]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.42, 1.0), ncol=len(handles), frameon=False,
              fontsize=6.3, handletextpad=0.45, handlelength=1.7, columnspacing=1.1, borderpad=0.1)
    return pd.DataFrame(rows)


def line_style(rule):
    ref = rule in REFERENCE_RULES
    if rule == "oracle":
        return dict(color=RULE_COLORS[rule], linewidth=0.9, linestyle=(0, (1, 1.5)), zorder=3)
    return dict(color=RULE_COLORS[rule], linewidth=1.25 if ref else 1.55, linestyle=(0, (4, 2)) if ref else "-",
                zorder=5 if rule == "population" else 4)


def marker_style(rule, size=4.2):
    ref = rule in REFERENCE_RULES
    color = RULE_COLORS[rule]
    return dict(marker="o", markersize=size, markerfacecolor="white" if ref else color,
                markeredgecolor=color if ref else "white", markeredgewidth=1.0 if ref else 0.6)


def draw_gain_panel(ax, curves, match):
    table = []
    for rule in RULES_BC:
        g = curves[curves["strategy"].eq(rule)].sort_values("n_treated")
        x = 100 * g["actual_cell_share"].to_numpy(float)
        y = 100 * g["avoided_share_of_baseline"].to_numpy(float)
        ax.plot(x, y, **line_style(rule))
        table.append(pd.DataFrame(dict(rule=rule, nominal_budget=g["nominal_budget_fraction"].to_numpy(),
                                       n_cells_cooled=g["n_treated"].to_numpy(), cell_share_pct=x,
                                       avoided_share_of_baseline_pct=y)))
    m = match.set_index("strategy")
    ref = m.loc["population"]
    x_ref = 100 * ref["reference_n_treated"] / ref["n_cells"]
    y_ref = 100 * ref["target_avoided_share_of_baseline"]
    others = [r for r in RULES_BC if r not in ("oracle", "population") and m.loc[r, "status"] == "reached"]
    needed = 100 * m.loc[others, "actual_cell_share_needed"].astype(float)
    attained = 100 * m.loc[others, "attained_avoided_share_of_baseline"].astype(float)
    x_far = max([x_ref] + list(needed))
    ax.hlines(y_ref, x_ref, x_far, color="#555555", linewidth=0.7, linestyle=(0, (3, 2)), zorder=2)
    for rule in others:
        ax.plot(needed[rule], attained[rule], linestyle="none", marker="o", markersize=3.9,
                markerfacecolor="white", markeredgecolor=RULE_COLORS[rule], markeredgewidth=1.0, zorder=7)
    ax.plot(x_ref, y_ref, linestyle="none", zorder=8, **marker_style("population", 5.2))
    ax.annotate(f"{y_ref:.0f}% avoided by\ncooling the most\npopulous {x_ref:.0f}% of cells", xy=(x_ref, y_ref),
                xytext=(0.035, 0.975), textcoords="axes fraction", fontsize=6.3, color=TEXT, ha="left", va="top",
                linespacing=1.15,
                arrowprops=dict(arrowstyle="-", color="#777777", linewidth=0.5, shrinkA=2, shrinkB=3,
                                relpos=(0.45, 0.0)))
    ax.set_xlim(0, 101)
    ax.set_ylim(0, None)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xlabel("Grid cells cooled (% of eligible cells)", fontsize=7.5)
    ax.set_ylabel("Exposure avoided\n(% of baseline exposure)", fontsize=7.5)
    handles = [Line2D([0], [0], label=RULE_LABELS[r], **line_style(r)) for r in RULES_BC]
    ax.legend(handles=handles, loc="lower right", bbox_to_anchor=(1.0, 0.0), frameon=False, fontsize=6.2,
              handlelength=2.1, handletextpad=0.5, labelspacing=0.28, borderpad=0.2)
    style_axis(ax)
    return pd.concat(table, ignore_index=True)


def draw_magnitude_panel(ax, magnitude):
    table = magnitude.pivot(index="strategy", columns="delta_t_C", values="avoided_share_of_baseline").loc[RULES_BC] * 100
    deltas = table.columns.to_numpy(float)
    for rule in RULES_BC:
        ax.plot(deltas, table.loc[rule], **line_style(rule))
        if rule != "oracle":
            ax.plot(deltas, table.loc[rule], linestyle="none", zorder=6, **marker_style(rule))
    pad = 0.18 * (deltas.max() - deltas.min())
    ax.set_xlim(deltas.min() - pad, deltas.max() + pad)
    ax.set_ylim(0, None)
    ax.set_xticks(deltas)
    ax.set_xticklabels([f"{d:g}" for d in deltas])
    ax.set_xlabel("Cooling magnitude (\u00b0C)", fontsize=7.5)
    ax.set_ylabel("Exposure avoided\n(% of baseline exposure)", fontsize=7.5)
    style_axis(ax)
    out = table.copy()
    out.columns = [f"avoided_pct_at_{c:g}C" for c in out.columns]
    return out.reset_index().rename(columns={"strategy": "rule"})


# -----------------------------------------------------------------------------
# 5. Plotting
# -----------------------------------------------------------------------------
rules_data, split, years = load_rules()
curves, magnitude, match = load_gain(split)

fig = plt.figure(figsize=(7.2, 5.5), dpi=200)
ax_a = fig.add_axes([0.165, 0.095, 0.290, 0.810])
ax_b = fig.add_axes([0.620, 0.560, 0.360, 0.345])
ax_c = fig.add_axes([0.620, 0.095, 0.360, 0.345])

rules_table = draw_rules_panel(ax_a, rules_data)
gain_table = draw_gain_panel(ax_b, curves, match)
magnitude_table = draw_magnitude_panel(ax_c, magnitude)

for label, x, y in [("(a)", 0.010, 0.985), ("(b)", 0.500, 0.985), ("(c)", 0.500, 0.500)]:
    fig.text(x, y, label, fontsize=10, fontweight="bold", color=TEXT, ha="left", va="top")

for ext, kw in [("png", dict(dpi=600)), ("pdf", {}), ("svg", {})]:
    fig.savefig(f"{OUT_STEM}.{ext}", facecolor="white", **kw)
rules_table.to_csv(f"{OUT_STEM}_panel_a_rules.csv", index=False)
rules_data[["city", "heatwave_type", "strategy", "relative_efficiency", "avoided_exposure",
            "oracle_avoided"]].to_csv(f"{OUT_STEM}_panel_a_city_values.csv", index=False)
gain_table.to_csv(f"{OUT_STEM}_panel_b_gain_curves.csv", index=False)
magnitude_table.to_csv(f"{OUT_STEM}_panel_c_magnitude.csv", index=False)

plt.show()
print("Output:", OUTDIR)